# Appendix A: Metrics and Null Vectors

Printed source span verified before authoring: Appendix A is listed as printed pages 585-588. The PDF text layer contains printed pages 585-587 on PDF pages 602-604; printed page 588 appears to be an omitted blank page before Appendix B. This notebook is an original reference treatment of the same topic. It does not reproduce the appendix; it turns the appendix into executable habits: choose a metric, classify its signature, compute with the bilinear form, and test null behavior before trusting a model.

The purpose is practical. In geometric algebra, the outer product can tell us how subspaces are spanned, but metric products need a rule for measurement. The moment an inner product enters, signs and degeneracies become part of the geometry. A Euclidean vector with zero square is the zero vector. A mixed-signature vector with zero square can be a nonzero null vector. A degenerate basis vector can also square to zero, but for a different reason: the metric has thrown away measurement in that direction. Those cases should not be collapsed in code.


## Translation Guide

Read the appendix vocabulary as a small implementation contract. A bilinear form is the scalar-valued measurement function. In a diagonal basis we can write it as a signed coordinate sum, but the coordinate sum is not the concept; it is just a convenient representation after diagonalization. The signature counts how many independent unit directions have positive square and how many have negative square. The optional third count in this notebook is the degenerate count, where basis vectors have square zero because the metric has no inverse in those directions.

Null vectors are nonzero vectors whose square is zero. In a mixed metric they usually arrive in reciprocal pairs. A reciprocal pair is not orthogonal in the usual Euclidean sense: each vector is null, but their mutual inner product can be nonzero. This is exactly why conformal geometric algebra can use null directions to encode Euclidean points, infinity, and origins without treating zero length as zero information.

The warning about rotors in general metrics is also a software warning. A rotor implementation that works in Euclidean space may silently assume a positive norm, a connected exponential parameterization, or a harmless reverse. Those assumptions should be asserted when a library is meant to support non-Euclidean or degenerate signatures.


## Route

We move through the appendix in five compact passes. First, we record the source-span audit as an artifact so future readers can see which printed pages were checked. Second, we build a metric table that separates Euclidean, Minkowski, conformal-style, and degenerate signatures. Third, we visualize Gram matrices so sign and rank are visible at a glance. Fourth, we construct null vectors and null blades, including the common pitfall that a blade containing null factors need not itself be null. Finally, we run a small metric scan that classifies vectors by the sign of their square and writes a sanity report.

The code uses the local dictionary-backed algebra in `utils.ga`. That implementation assumes an orthogonal diagonal metric, which is exactly the right test bench for Appendix A: it keeps the focus on signature, basis squares, and grade behavior rather than on a full symbolic algebra system.


In [1]:
from pathlib import Path
import json, math, sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
BOOK_ROOT = Path.cwd()
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils" / "ga").exists():
        BOOK_ROOT = candidate
        break
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))
from utils.ga import Algebra
from utils.appendix_reference import (
    APPENDIX_ARTIFACT_ROOT, bilinear, blade_square, gram_matrix, inner_scalar,
    metric_signature, minkowski_null_pair, quadratic_form, source_span_records,
    write_csv_artifact, write_json_artifact,
)
np.set_printoptions(precision=5, suppress=True)
pd.set_option("display.max_columns", 12)
APPENDIX_KEY = "appendix-a"
artifact_paths = {}
def remember(label, path):
    artifact_paths[label] = str(Path(path)); return Path(path)
print("project", BOOK_ROOT)
print("appendix artifacts", APPENDIX_ARTIFACT_ROOT)


project Geometric-Algebra-for-Computer-Science
appendix artifacts Geometric-Algebra-for-Computer-Science/artifacts/appendices


In [2]:
source_rows = source_span_records()
source_payload = {
    "pdf": "Geometric Algebra for Computer Science.pdf",
    "verification_tools": ["pdfinfo", "pdftotext -layout"],
    "pdf_page_count": 640,
    "note": "Printed pages are authoritative; omitted blank pages are listed explicitly.",
    "records": source_rows,
}
remember("source_span_json", write_json_artifact(["source"], "source-span-audit.json", source_payload))
remember("source_span_csv", write_csv_artifact(["source"], "source-span-audit.csv", source_rows))
pd.DataFrame(source_rows)


,appendix,title,printed_span,text_pdf_pages,missing_or_blank_printed_pages,sections
0,A,Metrics and Null Vectors,585-588,"[602, 603, 604]",[588],"[A.1 bilinear form, A.2 diagonalization to ort..."
1,B,Contractions and Other Inner Products,589-596,"[605, 606, 607, 608, 609, 610, 611, 612]",[],"[B.1 other inner products, B.2 equivalence of ..."
2,C,Subspace Products Retrieved,597-602,"[613, 614, 615, 616, 617]",[602],"[C.1 outer product from geometric product, C.2..."
3,D,Common Equations,603-608,"[618, 619, 620, 621, 622]",[608],"[D.1 product and operation notation, D.2 sign ..."


## Metrics As Executable Objects

A metric is easy to under-specify in prose. The executable version is more disciplined: every example gets a diagonal metric vector, a signature, a degeneracy flag, and a few probe vectors. The probes are not random decoration. They catch three common mistakes: treating `R^n` as if it always meant Euclidean space, forgetting that mixed metrics can produce positive, negative, and zero squares, and assuming that a zero square always means the vector is structurally zero.

The table below deliberately includes a degenerate metric. Degenerate algebras are awkward because blades in the degenerate direction often lack inverses. That awkwardness is useful here because it forces a distinction between a null vector produced by cancellation in a nondegenerate mixed metric and a basis vector whose square is zero because the metric itself has rank loss.


In [3]:
metric_examples = {
    "Euclidean R3,0": ([1, 1, 1], [(1, 2, 3), (0, 1, -1)]),
    "Minkowski R1,1": ([-1, 1], [(1, 1), (2, 1), (1, 2)]),
    "Conformal-style R4,1 diagonal": ([1, 1, 1, 1, -1], [(0, 0, 0, 1, 1), (1, 2, 0, 1, 1)]),
    "Degenerate R2 plus null axis": ([1, 1, 0], [(1, 0, 3), (0, 0, 5)]),
    "Balanced mixed R2,2": ([1, 1, -1, -1], [(1, 0, 1, 0), (1, 2, 0, 1)]),
}
metric_rows = []
for name, (metric, probes) in metric_examples.items():
    sig = metric_signature(metric)
    for probe in probes:
        metric_rows.append({
            "example": name, "metric": metric, "signature": sig["signature"],
            "nondegenerate": sig["nondegenerate"], "probe": probe,
            "square": quadratic_form(probe, metric),
        })
remember("signature_examples_csv", write_csv_artifact([APPENDIX_KEY, "tables"], "signature-examples.csv", metric_rows))
pd.DataFrame(metric_rows)


,example,metric,signature,nondegenerate,probe,square
0,"Euclidean R3,0","[1, 1, 1]","R^3,0",True,"(1, 2, 3)",14.0
1,"Euclidean R3,0","[1, 1, 1]","R^3,0",True,"(0, 1, -1)",2.0
2,"Minkowski R1,1","[-1, 1]","R^1,1",True,"(1, 1)",0.0
3,"Minkowski R1,1","[-1, 1]","R^1,1",True,"(2, 1)",-3.0
4,"Minkowski R1,1","[-1, 1]","R^1,1",True,"(1, 2)",3.0
5,"Conformal-style R4,1 diagonal","[1, 1, 1, 1, -1]","R^4,1",True,"(0, 0, 0, 1, 1)",0.0
6,"Conformal-style R4,1 diagonal","[1, 1, 1, 1, -1]","R^4,1",True,"(1, 2, 0, 1, 1)",5.0
7,Degenerate R2 plus null axis,"[1, 1, 0]","R^2,0 with 1 degenerate",False,"(1, 0, 3)",1.0
8,Degenerate R2 plus null axis,"[1, 1, 0]","R^2,0 with 1 degenerate",False,"(0, 0, 5)",0.0
9,"Balanced mixed R2,2","[1, 1, -1, -1]","R^2,2",True,"(1, 0, 1, 0)",0.0


## Gram Matrices And Rank

A Gram matrix is the metric seen through a chosen list of vectors. It is the fastest compact diagnostic for Appendix A. If the matrix has full rank and mixed signs, the space is nondegenerate but indefinite. If the matrix loses rank, some measurement has become invisible to the selected span. In numerical work those two situations can look similar if you only ask whether a particular vector has square zero.

The plot uses a small mixed metric and four probe vectors. The diagonal entries show vector squares; the off-diagonal entries show mutual inner products. A null vector can have a zero diagonal entry while still pairing nontrivially with another vector. That is the point of the reciprocal null basis used throughout conformal geometric algebra.


In [4]:
metric = [-1, 1, 1]
probes = np.array([[1, 0, 0], [0, 1, 0], [1, 1, 0], [1, -1, 1]], dtype=float)
G = gram_matrix(probes, metric)
eigenvalues = np.linalg.eigvalsh(G)
fig, ax = plt.subplots(figsize=(5.2, 4.2))
limit = max(abs(G).ravel())
im = ax.imshow(G, cmap="coolwarm", vmin=-limit, vmax=limit)
ax.set_xticks(range(len(probes)), labels=[f"v{i}" for i in range(len(probes))])
ax.set_yticks(range(len(probes)), labels=[f"v{i}" for i in range(len(probes))])
for i in range(G.shape[0]):
    for j in range(G.shape[1]):
        ax.text(j, i, f"{G[i, j]:.0f}", ha="center", va="center", color="black")
ax.set_title("Gram matrix under diag(-1, 1, 1)")
fig.colorbar(im, ax=ax, shrink=0.8)
fig.tight_layout()
figure_path = remember("gram_matrix_png", APPENDIX_ARTIFACT_ROOT / APPENDIX_KEY / "figures" / "mixed-gram-matrix.png")
figure_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(figure_path, dpi=160)
plt.close(fig)
gram_payload = {"metric": metric, "gram": G.tolist(), "eigenvalues": eigenvalues.tolist()}
remember("gram_matrix_json", write_json_artifact([APPENDIX_KEY, "checks"], "gram-matrix.json", gram_payload))
gram_payload


{'metric': [-1, 1, 1],
 'gram': [[-1.0, 0.0, -1.0, -1.0],
  [0.0, 1.0, 1.0, -1.0],
  [-1.0, 1.0, 0.0, -2.0],
  [-1.0, -1.0, -2.0, 1.0]],
 'eigenvalues': [-2.7177411174640036,
  4.516686194798963e-16,
  0.325396771834382,
  3.392344345629622]}

## Null Vectors

The cleanest null-vector check uses the two-dimensional Minkowski plane. With one negative and one positive basis square, the diagonal terms can cancel. The two null combinations below are reciprocal: each has zero square, but their mutual inner product is one. That combination is the small algebraic seed behind the origin and infinity basis used in conformal models.

Notice the implementation lesson. Testing `x.norm2() == 0` is not enough to decide that `x` can be discarded. A null vector can carry direction and incidence information. In a conformal model, most useful point representatives are null. Removing them because they have zero square would erase the geometry.


In [5]:
null_data = minkowski_null_pair()
checks = null_data["checks"]
assert abs(checks["n_plus_square"]) < 1e-12
assert abs(checks["n_minus_square"]) < 1e-12
assert abs(checks["mutual_inner"] - 1.0) < 1e-12
remember("minkowski_null_pair_json", write_json_artifact([APPENDIX_KEY, "checks"], "minkowski-null-pair.json", checks))
checks


{'n_plus_square': 0.0,
 'n_minus_square': 0.0,
 'mutual_inner': 0.9999999999999998,
 'pair_blade_square': 0.9999999999999996,
 'pair_blade': 'e_t^e_x'}

## Degenerate Nulls Are Different

A basis vector with square zero in a degenerate metric is also null, but not in the same operational sense as the reciprocal pair above. In the degenerate case there may be no vector that pairs with it to one inside the represented metric. That absence matters because reciprocal frames, duality, inverses, and projection formulas often require nondegeneracy.

The next cell compares a mixed-signature null vector with a degenerate basis null. The mixed vector has a partner. The degenerate basis vector does not acquire a reciprocal by cancellation; its measurement direction is missing. That is why algorithms should report both `square == 0` and the rank or degeneracy status of the metric.


In [6]:
mixed = minkowski_null_pair()
degenerate_alg = Algebra([1, 1, 0], names=["e1", "e2", "z"])
e1, e2, z = degenerate_alg.basis()
degenerate_checks = {
    "degenerate_metric": metric_signature([1, 1, 0]),
    "z_square": inner_scalar(z, z),
    "z_pairs_with_e1": inner_scalar(z, e1),
    "z_pairs_with_e2": inner_scalar(z, e2),
    "mixed_null_mutual_inner": mixed["checks"]["mutual_inner"],
}
remember("degenerate_null_json", write_json_artifact([APPENDIX_KEY, "checks"], "degenerate-null-comparison.json", degenerate_checks))
degenerate_checks


{'degenerate_metric': {'metric': (1.0, 1.0, 0.0),
  'dimension': 3,
  'positive': 2,
  'negative': 0,
  'zero': 1,
  'signature': 'R^2,0 with 1 degenerate',
  'nondegenerate': False,
  'determinant': 0.0},
 'z_square': 0.0,
 'z_pairs_with_e1': 0.0,
 'z_pairs_with_e2': 0.0,
 'mixed_null_mutual_inner': 0.9999999999999998}

## Null Blades And A Useful Trap

A composite blade can have zero square when one factor is null, but the converse is not a safe shortcut. The important trap in Appendix A is that a blade containing two reciprocal null vectors need not be null. The product of the reciprocal pair spans the same plane as the original positive and negative basis vectors, and its square is nonzero.

The test below constructs both cases in a three-dimensional mixed metric. One blade uses a single null factor and an ordinary Euclidean factor; it squares to zero. The other blade is the reciprocal null pair; it squares to a nonzero scalar. This is the sort of invariant that should live in a unit test for any implementation that exposes null basis elements.


In [7]:
alg3 = Algebra([-1, 1, 1], names=["t", "x", "y"])
t, x, y = alg3.basis()
n_plus = (t + x) / math.sqrt(2)
n_minus = (x - t) / math.sqrt(2)
single_null_blade = n_plus.wedge(y)
reciprocal_pair_blade = n_plus.wedge(n_minus)
null_blade_checks = {
    "single_null_blade": repr(single_null_blade),
    "single_null_blade_square": blade_square(single_null_blade),
    "reciprocal_pair_blade": repr(reciprocal_pair_blade),
    "reciprocal_pair_blade_square": blade_square(reciprocal_pair_blade),
}
assert abs(null_blade_checks["single_null_blade_square"]) < 1e-12
assert abs(null_blade_checks["reciprocal_pair_blade_square"] - 1.0) < 1e-12
remember("null_blades_json", write_json_artifact([APPENDIX_KEY, "checks"], "null-blades.json", null_blade_checks))
null_blade_checks


{'single_null_blade': '0.707107t^y + 0.707107x^y',
 'single_null_blade_square': 0.0,
 'reciprocal_pair_blade': 't^x',
 'reciprocal_pair_blade_square': 0.9999999999999996}

## Metric Scan, Pitfalls, And Takeaways

The final working check scans a grid of small integer vectors in a mixed metric and classifies them by the sign of the square. This catches the basic qualitative fact that Euclidean intuition is a special case: an indefinite metric partitions nonzero vectors into positive, negative, and null classes. In a production geometry library, a scan like this becomes a diagnostic around model setup. Before a conformal, projective, or physics-inspired model is used, the code should know whether the metric is nondegenerate, how many signs it has, and whether the intended null directions are present for the intended reason.

The main pitfall is to treat the metric as a passive annotation. It decides which vectors have inverses, which blades can be normalized, whether a square is a length, and whether a null representative is meaningful data. Store the signature with the algebra. Test the bilinear form directly. Distinguish cancellation-null vectors from degenerate metric directions. When a blade is null, ask why. When a formula uses an inverse, assert non-nullness before applying it.


In [8]:
scan_metric = [-1, 1, 1]
scan_rows = []
for a in range(-2, 3):
    for b in range(-2, 3):
        for c in range(-1, 2):
            if (a, b, c) == (0, 0, 0):
                continue
            square = quadratic_form((a, b, c), scan_metric)
            kind = "null" if abs(square) < 1e-12 else ("positive" if square > 0 else "negative")
            scan_rows.append({"coords": (a, b, c), "square": square, "kind": kind})
counts = pd.DataFrame(scan_rows).groupby("kind").size().to_dict()
remember("metric_scan_json", write_json_artifact([APPENDIX_KEY, "checks"], "metric-scan.json", {"counts": counts, "rows": scan_rows}))
counts


{'negative': 20, 'null': 12, 'positive': 42}

In [9]:
assert abs(checks["n_plus_square"]) < 1e-12
assert abs(checks["n_minus_square"]) < 1e-12
assert counts["null"] > 0 and counts["positive"] > 0 and counts["negative"] > 0
sanity_payload = {"appendix": "A", "artifact_count_before_sanity": len(artifact_paths), "artifacts": artifact_paths}
sanity_path = remember("final_sanity", write_json_artifact([APPENDIX_KEY, "checks"], "final-sanity.json", sanity_payload))
missing = [str(path) for path in map(Path, artifact_paths.values()) if not Path(path).exists()]
assert not missing, missing
assert Path(artifact_paths["source_span_json"]).exists()
print(json.dumps({"appendix": "A", "artifact_count": len(artifact_paths), "sanity": str(sanity_path)}, indent=2))


{
  "appendix": "A",
  "artifact_count": 10,
  "sanity": "Geometric-Algebra-for-Computer-Science//artifacts//appendices//appendix-a//checks//final-sanity.json"
}
